- 계층 손실(MCLoss) 그룹 항 1런 — 앵커 `11_01`에서 **손실만** 교체한 단일 변수 대조
- flat 188-way sigmoid + `Mno`→`Lno` 유도. 새 출력 유닛·게이트·임계 정책 변경 없음
- 기준선 test micro **0.8588**(11,244 · τ=0.5) · 운영 판정선 **+0.6pt = 0.8648**
- 주 위험은 empty rate 상승 예상(앵커 1.17%) — 음성 그룹 항이 로짓을 아래로 밀 수 있음
- 손실·지표는 `patent_train` 클래스를 상속·교체해 구현.

In [1]:
import json
from functools import partial

import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score

import sys
sys.path.insert(0, "src")  # /workspace/src

from patent_train import TrainingRunner, TrainConfig
from patent_train import losses as pt_losses
from patent_train.env import setup_env
from patent_train.losses import FocalLoss
from patent_train.metrics import sigmoid, f1_triple, empty_rate

setup_env()

In [2]:
SEARCH = False   # True=fast-fail 짧은 런(2 epoch로 유도) / False=풀런(아래 epochs 사용)

cfg = TrainConfig(
    backbone="axenc",   # backbones.BACKBONES 키
    loss="mcl",             # 아래 셀에서 LOSSES에 등록하는 키
    loss_params={"alpha": 0.25, "gamma": 2, "lam": None},   # lam은 λ 산출 셀에서 채운다
    max_len=512,
    eff_batch=128,          # 배치 재현 파라미터
    micro_batch=128,
    eval_micro_batch=512,
    learning_rate=4.8e-4,   # 확정 레시피 lr
    weight_decay=0.01,
    warmup_ratio=0.1,
    epochs=12,
    early_stop_epochs=2,    # 개선 없이 견디는 에폭 수(eval 횟수 환산은 runner가 처리)
    notebook_name="14_01_HierLoss_MCLoss.ipynb",   # wandb code saving
    tag="modernbert-patent-len512-mcl",
    run_name="axenc_len512_mcl_eff128_lr4.8e4_mcl",
    repo_final="ingyoun/A.X-patent-len512-mcl",
    out_path="/workspace/output/modernbert-len512-mcl",
    search=SEARCH,
)

print("run_name:", cfg.run_name, "| epochs:", cfg.epochs, "| grad_accum:", cfg.grad_accum)
print("loss:", cfg.loss, cfg.loss_params, "| seed:", cfg.seed)

run_name: axenc_len512_mcl_eff128_lr4.8e4_mcl | epochs: 12 | grad_accum: 1
loss: mcl {'alpha': 0.25, 'gamma': 2, 'lam': None} | seed: 42


## 손실 — MCLoss 그룹 항

C-HMCNN(Giunchiglia & Lukasiewicz, *Coherent Hierarchical Multi-Label Classification Networks*, NeurIPS 2020, [arXiv:2010.10151](https://arxiv.org/abs/2010.10151))의 **MCM**(상위 출력 = 하위 출력의 max)과 **MCLoss**를 `Lno`(17)→`Mno`(188) 2단 계층에 적용.

`Lno` 점수를 따로 두지 않는다. $y_L = \max_{m \in L} h_m$ 사용. 파라미터가 0개 늘고, MCM이 현행 `Mno`→`Lno` 유도 규칙과 수학적으로 동일해진다.

그룹 $L$당 항이 하나씩 걸린다(양성·음성 두 항 모두 둔다 — 음성 항만 남기면 로짓이 아래로만 밀려 k≥2 과소예측을 악화시킨다).

$$\ell_L = \begin{cases} -\log \big(\max_{m \in L,\, t_m = 1} \sigma(h_m)\big) & L\text{이 양성} \\ -\log\big(1 - \max_{m \in L} \sigma(h_m)\big) & L\text{이 음성} \end{cases}$$

$$\mathcal{L} = \mathcal{L}_{\text{focal}} + \lambda \cdot \mathcal{L}_{\text{group}}, \qquad \mathcal{L}_{\text{group}} = \frac{1}{17}\sum_L \ell_L$$

- **리덕션을 맞춘다.** focal이 요소 평균(문서×188)이므로 그룹 항도 문서당 17그룹 평균으로 둔다.
- **λ 고정** 새로운 HP를 튜닝하는 대신 $\lambda = \mathcal{L}_{\text{focal}}(\text{init}) / \mathcal{L}_{\text{group}}(\text{init})$로 고정.
- **원 논문과의 차이**: 논문은 모든 노드에 MCLoss를 적용해 BCE를 대체한다. 여기서는 188개 잎에 focal(γ=2)을 유지하고 17개 그룹 항을 추가한다.

In [3]:
# 17그룹 인덱스 — Mno 열 → Lno 그룹
lm = json.load(open(hf_hub_download("ingyoun/patent-clean-text", "label_mappings.json",
                                    repo_type="dataset"), encoding="utf-8"))
NUM_LABELS, N_GROUPS = 188, 17
mno_of_col = [lm["id2mno"][str(c)] for c in range(NUM_LABELS)]   # JSON 키는 문자열
lnos = sorted(set(lm["mno2lno"].values()))
LNO_IDX = np.array([lnos.index(lm["mno2lno"][m]) for m in mno_of_col])   # (188,) 열→그룹
M2L = np.zeros((NUM_LABELS, N_GROUPS), dtype=int)                        # (188,17) 사영
M2L[np.arange(NUM_LABELS), LNO_IDX] = 1

assert len(lnos) == N_GROUPS and M2L.sum() == NUM_LABELS
print(f"[group] {N_GROUPS}그룹 · 열 {NUM_LABELS} · 그룹 크기 {np.bincount(LNO_IDX).tolist()}")
print(f"[group] {lnos}")

label_mappings.json:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

[group] 17그룹 · 열 188 · 그룹 크기 [15, 8, 11, 11, 14, 7, 10, 15, 12, 11, 20, 15, 9, 10, 14, 4, 2]
[group] ['EA', 'EB', 'EC', 'ED', 'EE', 'EF', 'EG', 'EH', 'EI', 'LA', 'LB', 'LC', 'NB', 'NC', 'ND', 'OA', 'OB']


In [4]:
class MclFocalLoss(FocalLoss):
    """focal(188 mno) + λ·MCLoss 그룹 항(17 `Lno`).

    `y_L = max_{m∈L} h_m`
    별도 `Lno` 출력이 없고, MCM이 현행 `Mno`→`Lno` 유도 규칙과 항등이다.

        양성 L: -log σ(max_{m∈L, t=1} h_m)    정답 하나가 확신되면 포화
        음성 L: -log(1 - σ(max_{m∈L} h_m))    lno가 틀린 mno 억제

    리덕션은 focal과 동일하게 'mean'.
    """

    def __init__(self, *, lno_idx, lam, alpha: float = 0.25, gamma: int = 2):
        super().__init__(alpha=alpha, gamma=gamma)
        if lam is None:
            raise ValueError("lam 미설정 — 초기화 1배치로 산출한 값을 넣을 것")
        self.lam = float(lam)
        idx = torch.as_tensor(np.asarray(lno_idx), dtype=torch.long)   # scatter_reduce의 인덱스 텐서는 int64여야 함.
        self.n_groups = int(idx.max()) + 1
        self.register_buffer("lno_idx", idx, persistent=False)         # idx를 nn.Module에 텐서 등록하되 파라미터가 아니라고 표시. grad없이 학습과정에서 추적.

    def group_loss(self, logits, targets) -> torch.Tensor: # MCL의 L_group 구현
        z = logits.float()                                 # 그룹 항만 fp32. focal 경로는 손대지 않는다
        if self.lno_idx.device != z.device:                # loss_fn은 모델 서브모듈이 아니라 .to()가 안 걸린다
            self.lno_idx = self.lno_idx.to(z.device)
        n, neg = z.shape[0], torch.finfo(z.dtype).min
        idx = self.lno_idx.expand(n, -1)                   # lno_idx:(188,) -> expand : (n, 188)
        
        # MCL
        gz = z.new_full((n, self.n_groups), neg).scatter_reduce(dim=1, idx, z, "amax")  # (n, 17).gz[i][g]:문서i에서 그룹g에 속한 mno 로짓들의 최댓값->MCM
        
        # MCLoss 양성항 마스킹
        gz_pos = z.new_full((n, self.n_groups), neg).scatter_reduce(
            1, idx, z.masked_fill(targets <= 0, neg), "amax")       # 정답이 아닌 mno 로짓값을 neg로 마스킹
        
        # 그룹 정답
        gt = z.new_zeros((n, self.n_groups)).scatter_reduce(1, idx, targets.float(), "amax")  # (n, 17). 정답 벡터
        pos = gt > 0   # where에서 쓸 조건식
        
        h = torch.where(pos, gz_pos, gz)   # 양성=정답 중 max · 음성=전체 중 max. 오답 Lno에 속한 것 중 가장 큰 mno 로짓을 아래로 누름.
        return torch.where(pos, -F.logsigmoid(h), -F.logsigmoid(-h)).mean()

    def forward(self, logits, targets) -> torch.Tensor:
        # group_loss는 Lno를 잘못 짚은 오류를 겨냥하고, Lno는 맞혔는데 형제를 헷갈린 오류는 focal에 맡긴다. 
        # 한 그룹(Lno)에 정답(Mno)가 복수라면 group loss의 양성항은 둘 중 높은 값에만 신호를 줘 더 키우고 낮은 쪽은 group loss의 신호를 받지 못한다
        return super().forward(logits, targets) + self.lam * self.group_loss(logits, targets) # L_focal + Lambda * L_group


# 레지스트리 등록 — runner.build_trainer가 build_loss(cfg.loss, **cfg.loss_params)로 호출
pt_losses.LOSSES["mcl"] = partial(MclFocalLoss, lno_idx=LNO_IDX)
print("LOSSES:", sorted(pt_losses.LOSSES))

LOSSES: ['asl', 'bce', 'focal', 'mcl', 'zlpr']


## eval 지표 — `Lno` 유도 축·표적 질량 병기

`patent_train.metrics`의 5개 키(`micro_f1`·`macro_f1`·`sample_f1`·`empty_rate`·`anchor_weighted_f1`)를 유지하고 7개를 더해, 훈련 중 개선 여부와 **기제**를 훈련 곡선에서 관찰.

| 키 | 무엇인가 | 앵커 `11_01` |
| --- | --- | --- |
| `lno_micro_f1`·`lno_macro_f1`·`lno_sample_f1` | `Mno`→`Lno` 유도 축(17) F1 | — |
| `fp_cross_lno_share` | 정답 `Lno` **밖**에 떨어진 FP 비율 — **음성 항의 표적** | 0.604 |
| `fn_group_missed_share` | 그룹 정답을 하나도 못 맞힌 FN 비율 — **양성 항의 표적** | 0.780 |
| `fp_per_doc`·`fn_per_doc` | 문서당 FP·FN 절대량 | 0.178 · 0.164 |

`f1_triple`·`sigmoid`·`empty_rate`는 `patent_train.metrics`에서 그대로 재사용.

**`metric_for_best_model="micro_f1"`** — 체크포인트 선택, 판정도 `micro_f1` 유지.`lno_micro_f1`이 올라도 `micro_f1`이 안 오르면 이득이 아니다(주 지표는 `Mno` 축).

In [ ]:
def make_compute_metrics_lno(tau, lno_idx=LNO_IDX, m2l=M2L):
    """patent_train.metrics.make_compute_metrics + Lno 유도 축 + MCLoss 두 항의 표적 질량."""

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        logits = np.asarray(logits)
        Y = np.asarray(labels).astype(bool)
        P = sigmoid(logits) >= tau
        Yl, Pl = (Y.astype(int) @ m2l) > 0, (P.astype(int) @ m2l) > 0    # Mno→Lno 유도
        tp, fp, fn = P & Y, P & ~Y, ~P & Y
        in_gold_grp = Yl[:, lno_idx]                        # 열 c의 Lno가 정답 Lno 집합에 있나
        grp_hit = ((tp.astype(int) @ m2l) > 0)[:, lno_idx]   # 그 Lno에서 정답을 하나라도 맞혔나
        n_fp, n_fn = int(fp.sum()), int(fn.sum())
        return {
            **f1_triple(Y.astype(int), P.astype(int)),      # micro_f1 = 판정·best 기준(불변)
            "empty_rate": empty_rate(P.astype(int)),
            "anchor_weighted_f1": f1_score(Y.argmax(1), logits.argmax(1),
                                           average="weighted", zero_division=0),
            **{f"lno_{k}": v for k, v in f1_triple(Yl.astype(int), Pl.astype(int)).items()},
            "fp_cross_lno_share": float((fp & ~in_gold_grp).sum() / max(n_fp, 1)),   # 음성 항 표적
            "fn_group_missed_share": float((fn & ~grp_hit).sum() / max(n_fn, 1)),    # 양성 항 표적
            "fp_per_doc": n_fp / len(Y),
            "fn_per_doc": n_fn / len(Y),
        }

    return compute_metrics


### verify
_c0, _c1, _cX = 0, 1, NUM_LABELS - 1
assert LNO_IDX[_c0] == LNO_IDX[_c1] and LNO_IDX[_cX] != LNO_IDX[_c0]
_Y = np.zeros((4, NUM_LABELS), dtype=int)
_Z = np.full((4, NUM_LABELS), -10.0)
_Y[0, _c0] = 1; _Z[0, _c0] = 10.0                              # 오류 없음
_Y[1, _c0] = 1; _Z[1, _c0] = 10.0; _Z[1, _c1] = 10.0           # FP 1건 · 정답 Lno "안"(형제)
_Y[2, _c0] = 1; _Z[2, _cX] = 10.0                              # FP 1건 Lno "밖" + FN 1건 그룹 놓침
_Y[3, _c0] = _Y[3, _c1] = 1; _Z[3, _c0] = 10.0                 # FN 1건이나 형제 적중으로 max 포화

_m = make_compute_metrics_lno(0.5)((_Z, _Y))
assert _m["fp_per_doc"] == 0.5 and _m["fn_per_doc"] == 0.5     # FP 2건 · FN 2건 / 4문서
assert abs(_m["fp_cross_lno_share"] - 0.5) < 1e-9              # 2건 중 문서2만 Lno 밖
assert abs(_m["fn_group_missed_share"] - 0.5) < 1e-9           # 2건 중 문서2만 그룹 전체 놓침
assert _m["lno_micro_f1"] > _m["micro_f1"] and _m["empty_rate"] == 0.0
print(f"[verify] 지표 소형 케이스 통과 — FP cross {_m['fp_cross_lno_share']:.3f}"
      f" · FN missed {_m['fn_group_missed_share']:.3f}"
      f" · micro {_m['micro_f1']:.4f} → lno {_m['lno_micro_f1']:.4f}")
print(f"[verify] 지표 키 {len(_m)}개: {sorted(_m)}")

[verify] 지표 소형 케이스 통과 — FP cross 0.500 · FN missed 0.500 · micro 0.6000 → lno 0.7500
[verify] 지표 키 12개: ['anchor_weighted_f1', 'empty_rate', 'fn_group_missed_share', 'fn_per_doc', 'fp_cross_lno_share', 'fp_per_doc', 'lno_macro_f1', 'lno_micro_f1', 'lno_sample_f1', 'macro_f1', 'micro_f1', 'sample_f1']


## 구성 — 데이터·모델

In [7]:
runner = TrainingRunner(cfg)

In [8]:
runner.load_data()        # 토크나이저 + 원본 데이터셋(prep 캐시 있으면 원본 생략)
runner.data.raw

[skip] prep 캐시 존재 — 원본 로드 생략: /workspace/prep_cache/axenc_len512


In [9]:
runner.prepare_data()     # max_len 절단 → prep 캐시
runner.data.dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 201616
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11244
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11132
    })
})

In [10]:
runner.load_model()

[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForSequenceClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[model] skt/A.X-Encoder-base@9708f9c4(신규 헤드)


## λ 산출 — 초기화 1배치 forward

$\lambda = \mathcal{L}_{\text{focal}}(\text{init}) / \mathcal{L}_{\text{group}}(\text{init})$. 

`build_trainer()` 전에 λ 산출. `build_trainer`가 `build_loss`를 호출하는 시점에 `lam`이 채워져 있어야 하고, 비어 있으면 `MclFocalLoss.__init__`이 `ValueError` 발생.

In [11]:
# load_model() 직후 모델은 CPU에 있고 flash-attention-2는 CUDA 전용이라 이동이 필요하다
# 두 항은 fp32로 재서 bf16 반올림이 λ에 섞이지 않게 한다.
# 배치는 등간격 인덱스로 뽑는다. 등간격은 난수를 쓰지 않아 재현되고 17그룹 양성·음성 구성이 고루 섞인다.
_ds = runner.data.dataset["train"]
_pick = np.linspace(0, len(_ds) - 1, cfg.micro_batch, dtype=int)
_batch = runner.data.collator([_ds[int(i)] for i in _pick])
runner.model.to("cuda").eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    _logits = runner.model(**{k: v.to("cuda") for k, v in _batch.items() if k != "labels"}).logits
_t_lam = _batch["labels"].float().to("cuda")

L_FOCAL = float(FocalLoss(alpha=0.25, gamma=2)(_logits.float(), _t_lam))
L_GROUP = float(MclFocalLoss(lno_idx=LNO_IDX, lam=0.0).group_loss(_logits.float(), _t_lam))
LAM = L_FOCAL / L_GROUP
cfg.loss_params["lam"] = LAM

_pos_grp = (_t_lam.cpu().numpy().astype(int) @ M2L) > 0
print(f"[lambda] batch={cfg.micro_batch}(등간격) · 문서당 양성 Lno 평균 {_pos_grp.sum(1).mean():.3f}"
      f" · 등장 Lno {int(_pos_grp.any(0).sum())}/{N_GROUPS}")
print(f"[lambda] focal={L_FOCAL:.6f} · group={L_GROUP:.6f}")
print(f"[lambda] lambda = focal/group = {LAM:.6f}")

[lambda] batch=128(등간격) · 문서당 양성 Lno 평균 1.086 · 등장 Lno 17/17
[lambda] focal=0.065730 · group=1.480618
[lambda] lambda = focal/group = 0.044393


## 훈련

In [12]:
runner.build_trainer()
runner.trainer.compute_metrics = make_compute_metrics_lno(cfg.tau)   # Lno 축·표적 질량 병기

### 주입 확인
assert isinstance(runner.trainer.loss_fn, MclFocalLoss), type(runner.trainer.loss_fn)
assert runner.trainer.loss_fn.lam == LAM and runner.trainer.loss_fn.n_groups == N_GROUPS
assert runner.trainer.loss_fn.alpha == 0.25 and runner.trainer.loss_fn.gamma == 2
assert runner.trainer.args.metric_for_best_model == "micro_f1"   # 선택 축 불변
print(f"[loss] {type(runner.trainer.loss_fn).__name__}(alpha=0.25, gamma=2, lam={LAM:.6f},"
      f" {N_GROUPS}그룹) · best 기준 {runner.trainer.args.metric_for_best_model}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


[schedule] 1576 step/epoch | eval·save 788 step마다(2회/epoch) | early stop 2 epoch(patience=4 eval)
[loss] MclFocalLoss(alpha=0.25, gamma=2, lam=0.044393, 17그룹) · best 기준 micro_f1


In [13]:
runner.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1,Lno Micro F1,Lno Macro F1,Lno Sample F1,Fp Cross Lno Share,Fn Group Missed Share,Fp Per Doc,Fn Per Doc
788,0.006469,0.004850,0.660585,0.627065,0.618774,0.187657,0.650810,0.769974,0.775764,0.712154,0.365344,0.901597,0.258175,0.483830
1576,0.004687,0.004048,0.718133,0.712188,0.720924,0.067284,0.699450,0.825843,0.829948,0.812361,0.419756,0.859803,0.361031,0.328063
2364,0.004087,0.003653,0.739947,0.729325,0.736012,0.078692,0.725221,0.841914,0.841785,0.821771,0.387775,0.844166,0.273356,0.337226
3152,0.003625,0.003511,0.761785,0.749508,0.752897,0.082824,0.741994,0.843402,0.846290,0.821212,0.443065,0.846030,0.209845,0.334890
3940,0.003370,0.003075,0.779081,0.770264,0.780730,0.059289,0.762209,0.861818,0.865684,0.849615,0.453406,0.817172,0.214966,0.299227
4728,0.003191,0.003000,0.779141,0.773406,0.793158,0.035843,0.761551,0.867369,0.866833,0.865938,0.433186,0.811745,0.264193,0.267697
5516,0.002880,0.002844,0.793933,0.788271,0.809051,0.033507,0.775459,0.877057,0.880423,0.876852,0.441834,0.794737,0.237064,0.256019
6304,0.002783,0.002782,0.799455,0.794420,0.805995,0.044017,0.781776,0.875351,0.877426,0.868659,0.429042,0.822052,0.218379,0.257456
7092,0.002444,0.002762,0.807552,0.802296,0.819763,0.029554,0.778085,0.883980,0.886836,0.882811,0.432391,0.814453,0.235178,0.229968
7880,0.002418,0.002686,0.806648,0.801604,0.821695,0.028476,0.782420,0.883588,0.887978,0.884601,0.450222,0.791246,0.222871,0.240119


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 평가 · 메트릭 저장 · push

모델 가중치는 로컬에 두지 않고 Hub로만 올린다(팟을 지우면 로컬 사본은 사라진다). 로컬 사본이 필요하면 `runner.save_model()`.

In [14]:
test_metrics = runner.evaluate("test")   # 앵커 11_01(0.8588) 대비 · 판정선 0.8648
for k, v in test_metrics.items():
    print(f"{k}: {v}")

[transformers] early stopping required metric_for_best_model, but did not find eval_micro_f1 so early stopping is disabled


Training Loss,Validation Loss,Step,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1,Lno Micro F1,Lno Macro F1,Lno Sample F1,Fp Cross Lno Share,Fn Group Missed Share,Fp Per Doc,Fn Per Doc
0.000212,0.003649,18912,0.846743,0.844323,0.868540,0.006137,0.814535,0.907422,0.910619,0.917986,0.552991,0.760511,0.208111,0.167111


test_loss: 0.003649085061624646
test_micro_f1: 0.8467434341966653
test_macro_f1: 0.8443232204550342
test_sample_f1: 0.8685398917656122
test_empty_rate: 0.0061366061899679825
test_anchor_weighted_f1: 0.8145348037077924
test_lno_micro_f1: 0.9074215761285387
test_lno_macro_f1: 0.9106193689393214
test_lno_sample_f1: 0.9179861711078359
test_fp_cross_lno_share: 0.552991452991453
test_fn_group_missed_share: 0.7605109100585418
test_fp_per_doc: 0.20811099252934898
test_fn_per_doc: 0.16711134827463536


In [15]:
runner.save_metrics()     # runner.metrics(split 전체) → {tag}_metrics.json

[save] /workspace/output/modernbert-len512-mcl/modernbert-patent-len512-mcl_metrics.json  splits=['test']


In [16]:
runner.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

[push] ingyoun/A.X-patent-len512-mcl


## val·test 로짓 덤프

`logits_{tag}_{split}.npy`를 `out_path` 상위(`/workspace/output/`)에 저장. 덤프 동안 샘플러가 순차로 복귀하고 반환 라벨이 데이터셋과 행 단위로 대조된다(`13_02`의 순열 사고 방어).

In [17]:
runner.predict_logits("val")
runner.predict_logits("test")

[dump] /workspace/output/logits_modernbert-patent-len512-mcl_val.npy  shape=(11132, 188)


[dump] /workspace/output/logits_modernbert-patent-len512-mcl_test.npy  shape=(11244, 188)


array([[ -1.3515625,  -4.3125   ,  -5.21875  , ..., -10.       ,
         -9.625    , -10.1875   ],
       [  5.       ,  -4.84375  ,  -4.125    , ...,  -9.9375   ,
        -11.6875   ,  -9.8125   ],
       [  4.90625  ,  -5.25     ,  -3.75     , ..., -11.6875   ,
        -12.3125   , -11.1875   ],
       ...,
       [ -9.3125   ,  -9.0625   , -11.1875   , ..., -10.5625   ,
         -6.34375  ,   8.625    ],
       [ -9.75     ,  -9.8125   , -11.375    , ..., -11.75     ,
         -7.1875   ,   7.40625  ],
       [ -9.1875   , -10.       , -11.5625   , ..., -10.625    ,
         -6.09375  ,   7.9375   ]], shape=(11244, 188), dtype=float32)